# Cleaning Notebook
This notebook demonstrates data cleaning workflows step-by-step for CGD dataset, including timestamp normalization, meter ID standardization, numeric conversions, and more.

In [4]:
import numpy as np
import pandas as pd


## Load raw dataset
Read the original dataset from CSV into a DataFrame for cleaning.

In [5]:
df = pd.read_csv('CGD_Dataset_before_cleaning.csv')

## Preserve original raw copy
Keep an untouched copy of raw data before making any transformations so that we can compare or restart cleaning.

In [6]:
df_raw = df.copy()

## Preview data table content
Displaying the DataFrame to ensure loading was successful.

In [7]:
df

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,"Latitude,Longitude",Leak_Flag,Maintenance_Flag,Notes_Comments,Base_Temp_C,Base_Pressure_kPa,Cal_Coefficient,Cal_Version
0,R9001,MET125,2025-06-05 00:00,5.18,3884,14.620,Sm3/h,C843,"13.0324,80.1744",N,NaN,Inspection done. No issues.,20.8,99.500,1.002,v1.0
1,R9002,MET-147,12/25/2024 14:00,6.87,781,2.930,Scmh,C726,"12.9747,80.2142",0,NaN,Email: user812@citygas.com flagged issue,21.9,101.325,1.000,v1.0
2,R9003,MET--132,2024-01-27 14:00,-3,793,3.130,Scmh,C756,"12.9182 , 80.3353",0,MAINT,NaN,21.3,101.000,1.000,v1.0
3,R9004,MET-131,2026-01-20 15:00,3.24,1130,4.460,m3/h,C807,"13.093,80.0978",no,MAINT,High consumption flagged for audit.,24.5,101.325,1.000,v1.1
4,R9005,met-108,2024-12-04 11:00,3.51,1434,5.020,Sm^3/h,C735,"13.1965,80.3709",nO,NaN,NaN,17.8,101.000,1.000,v1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6495,R15180,MET-105,2025-10-30 16:00,3.65,NaN,2.020,SCMH,C770,"12.9124,80.1624",FALSE,NaN,Calibration check passed.,19.9,101.325,1.000,v1.1
6496,R15181,MET-150,2026-03-05 09:00,4.68,NaN,1.840,SCMH,C724,"12.9239,80.0808",0,NaN,Customer called at +91-9853538-53538,15.7,101.325,1.000,v1.0
6497,R15182,met-128,2025-11-09 15:00,3.24,80,0.300,S m^3/h,C866,"13.0557,80.305",No,MAINT,Customer C752 complained about billing.,14.4,102.500,1.000,v1.1
6498,R15183,MET-151,06-04-2025 11:00,3.14,1753,6.150,scmh,C791,"13.2378,80.1659",nO,NaN,High consumption flagged for audit.,22.7,101.325,0.998,v1.0


## Data quality check
Compute basic metrics to understand shape, schema, and missing values before cleaning.

In [8]:
print("\n Shape:", df.shape)
print("\n Column names:\n", df.columns.tolist())
print("\n Data types:\n", df.dtypes)
print("\n Null counts:\n", df.isnull().sum())
print("\n First 5 rows:")
df.head()


 Shape: (6500, 16)

 Column names:
 ['Reading_ID', 'Meter_ID', 'TS', 'Pressure', 'Flow', 'Energy', 'Unit', 'Customer_ID', 'Latitude,Longitude', 'Leak_Flag', 'Maintenance_Flag', 'Notes_Comments', 'Base_Temp_C', 'Base_Pressure_kPa', 'Cal_Coefficient', 'Cal_Version']

 Data types:
 Reading_ID                str
Meter_ID                  str
TS                        str
Pressure                  str
Flow                      str
Energy                float64
Unit                      str
Customer_ID               str
Latitude,Longitude        str
Leak_Flag                 str
Maintenance_Flag          str
Notes_Comments            str
Base_Temp_C           float64
Base_Pressure_kPa     float64
Cal_Coefficient       float64
Cal_Version               str
dtype: object

 Null counts:
 Reading_ID               0
Meter_ID                 0
TS                       0
Pressure               247
Flow                   250
Energy                 260
Unit                     0
Customer_ID         

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,"Latitude,Longitude",Leak_Flag,Maintenance_Flag,Notes_Comments,Base_Temp_C,Base_Pressure_kPa,Cal_Coefficient,Cal_Version
0,R9001,MET125,2025-06-05 00:00,5.18,3884,14.62,Sm3/h,C843,"13.0324,80.1744",N,NaN,Inspection done. No issues.,20.8,99.500,1.002,v1.0
1,R9002,MET-147,12/25/2024 14:00,6.87,781,2.93,Scmh,C726,"12.9747,80.2142",0,NaN,Email: user812@citygas.com flagged issue,21.9,101.325,1.000,v1.0
2,R9003,MET--132,2024-01-27 14:00,-3,793,3.13,Scmh,C756,"12.9182 , 80.3353",0,MAINT,NaN,21.3,101.000,1.000,v1.0
3,R9004,MET-131,2026-01-20 15:00,3.24,1130,4.46,m3/h,C807,"13.093,80.0978",no,MAINT,High consumption flagged for audit.,24.5,101.325,1.000,v1.1
4,R9005,met-108,2024-12-04 11:00,3.51,1434,5.02,Sm^3/h,C735,"13.1965,80.3709",nO,NaN,NaN,17.8,101.000,1.000,v1.0


## Timestamp cleaning utility (Use Case 1)
Define a robust cleanup function for timestamp normalization across formats, timezone markers, and '24:' rollover.

In [9]:
# Use Case 1: Timestamp Normalization (ISO-first approach)

from datetime import timedelta

def fix_timestamp(ts_str):
    ts_str = str(ts_str).strip()

    # Step 1: Fix invalid hour 24:xx → roll over to next day
    if '24:' in ts_str:
        ts_str = ts_str.replace('24:', '00:')
        # Try ISO format first for the 24: case
        try:
            dt = pd.to_datetime(ts_str, format='%Y-%m-%d %H:%M', errors='raise')
        except:
            try:
                dt = pd.to_datetime(ts_str, format='%Y/%m/%d %H:%M', errors='raise')
            except:
                dt = pd.to_datetime(ts_str, errors='coerce')
        if pd.notnull(dt):
            dt = dt + timedelta(days=1)  # roll over to next day
        return dt

    # Step 2: Handle ISO format with timezone 'Z' (e.g. '2024-04-10T00:00:00Z')
    if 'T' in ts_str and ts_str.endswith('Z'):
        ts_str = ts_str.replace('Z', '')
        try:
            return pd.to_datetime(ts_str, format='%Y-%m-%dT%H:%M:%S', errors='raise')
        except:
            return pd.to_datetime(ts_str, errors='coerce')

    # Step 3: Try ISO format YYYY-MM-DD HH:MM (most common in our dataset)
    try:
        return pd.to_datetime(ts_str, format='%Y-%m-%d %H:%M', errors='raise')
    except:
        pass

    # Step 4: Try ISO with seconds YYYY-MM-DD HH:MM:SS
    try:
        return pd.to_datetime(ts_str, format='%Y-%m-%d %H:%M:%S', errors='raise')
    except:
        pass

    # Step 5: Try slash-separated ISO  YYYY/MM/DD HH:MM
    try:
        return pd.to_datetime(ts_str, format='%Y/%m/%d %H:%M', errors='raise')
    except:
        pass

    # Step 6: Try slash-separated American MM/DD/YYYY HH:MM (least preferred)
    try:
        return pd.to_datetime(ts_str, format='%m/%d/%Y %H:%M', errors='raise')
    except:
        pass

    # Final fallback — if nothing matched, return NaT
    return pd.to_datetime(ts_str, errors='coerce')


# Apply to column
df['TS'] = df['TS'].apply(fix_timestamp)

# Verify
failed = df['TS'].isnull().sum()
print(f"Timestamps cleaned. Failed to parse: {failed} rows")

print("\nSample timestamps after cleaning:")
print(df['TS'].head(10).to_string())

C:\Users\KIIT\AppData\Local\Temp\ipykernel_19908\3841325640.py:56: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(ts_str, errors='coerce')
C:\Users\KIIT\AppData\Local\Temp\ipykernel_19908\3841325640.py:56: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  return pd.to_datetime(ts_str, errors='coerce')


Timestamps cleaned. Failed to parse: 0 rows

Sample timestamps after cleaning:
0   2025-06-05 00:00:00
1   2024-12-25 14:00:00
2   2024-01-27 14:00:00
3   2026-01-20 15:00:00
4   2024-12-04 11:00:00
5   2024-10-19 12:00:00
6   2024-03-23 18:00:00
7   2024-10-03 00:00:00
8   2025-01-24 11:00:00
9   2024-08-02 22:00:00


In [10]:
# Use Case 2: Meter_ID Trimming, Uppercase, Hyphen Standardization

import re

def fix_meter_id(meter_id):
    mid = str(meter_id).strip()    # remove outer whitespace
    mid = mid.upper()              # uppercase all characters
    mid = mid.replace(' ', '')     # remove any internal spaces
    # re.sub(pattern, repl, string, count=0, flags=0)
    # Fix double or more hyphens → single hyphen
    mid = re.sub(r'-{2,}', '-', mid)

    # Fix missing hyphen: 'MET123' → 'MET-123'
    # Pattern: 'MET' followed directly by digits (no hyphen)
    mid = re.sub(r'^MET(\d+)$', r'MET-\1', mid)

    return mid

# Apply to the column
df['Meter_ID'] = df['Meter_ID'].apply(fix_meter_id)

# # Verify: check any Meter_IDs that still don't match expected format
# unexpected = df[~df['Meter_ID'].str.match(r'^MET-\d+$')]
# print(f"Meter_IDs not matching 'MET-XXX' format: {len(unexpected)}")
# if len(unexpected) > 0:
#     print(unexpected['Meter_ID'].unique())
# print("\nSample Meter_IDs after cleaning:")
print(df['Meter_ID'].head(10).to_string())

0    MET-125
1    MET-147
2    MET-132
3    MET-131
4    MET-108
5    MET-132
6    MET-107
7    MET-107
8    MET-130
9    MET-134


## Meter ID standardization (Use Case 2)
Convert Meter_ID to a cleaned canonical format (uppercase, no spaces, normalized hyphen).

In [11]:
# Use Case 3: Numeric Extraction for Pressure, Flow, Energy

def to_clean_number(value):
    """
    Converts a value to a clean float:
    - Removes commas used as thousand separators (e.g. '1,157' → 1157.0)
    - Strips whitespace from string numbers
    - Returns NaN if the value cannot be converted
    """
    if pd.isnull(value):
        return np.nan
    # Convert to string first, then clean
    val_str = str(value).strip().replace(',', '')
    try:
        return float(val_str)
    except ValueError:
        return np.nan  # if it still can't convert, return NaN

# Apply to all three numeric columns
df['Pressure'] = df['Pressure'].apply(to_clean_number)
df['Flow']     = df['Flow'].apply(to_clean_number)
df['Energy']   = df['Energy'].apply(to_clean_number)

# Check null counts after conversion (nulls = values that failed to parse)
print("Null counts after numeric conversion:")
print(f"  Pressure : {df['Pressure'].isnull().sum()}")
print(f"  Flow     : {df['Flow'].isnull().sum()}")
print(f"  Energy   : {df['Energy'].isnull().sum()}")

print("\nSample numeric values after cleaning:")
print(df[['Pressure', 'Flow', 'Energy']].head(10).to_string())

Null counts after numeric conversion:
  Pressure : 277
  Flow     : 250
  Energy   : 260

Sample numeric values after cleaning:
   Pressure    Flow  Energy
0      5.18  3884.0  14.620
1      6.87   781.0   2.930
2     -3.00   793.0   3.130
3      3.24  1130.0   4.460
4      3.51  1434.0   5.020
5      4.93  1219.0   4.620
6      4.31   744.0   2.910
7      5.43  1157.0   4.540
8      4.40  1188.0   4.556
9      2.76  4210.0  14.860


In [12]:
# Use Case 4: Unit Normalization → SCMH

# All known dirty variants found in this dataset mapped to 'SCMH'
unit_map = {
    'SCMH'    : 'SCMH',
    'scmh'    : 'SCMH',
    'Scmh'    : 'SCMH',
    'SCM/H'   : 'SCMH',
    'scm/h'   : 'SCMH',
    'Sm^3/h'  : 'SCMH',
    'sm^3/h'  : 'SCMH',
    'Sm3/h'   : 'SCMH',
    'm3/h'    : 'SCMH',
    'S m^3/h' : 'SCMH',  # has internal space
}

df['Unit'] = df['Unit'].astype(str).str.strip()
df['Unit'] = df['Unit'].map(unit_map)
unmapped = df['Unit'].isnull().sum()
print(f"Unmapped Unit values after normalization: {unmapped}")

df['Unit'] = df['Unit'].fillna('SCMH')


Unmapped Unit values after normalization: 0


In [13]:
df['Latitude,Longitude']

0           13.0324,80.1744
1           12.9747,80.2142
2        12.9182 , 80.3353 
3            13.093,80.0978
4           13.1965,80.3709
               ...         
6495        12.9124,80.1624
6496        12.9239,80.0808
6497         13.0557,80.305
6498        13.2378,80.1659
6499        13.1601,80.1145
Name: Latitude,Longitude, Length: 6500, dtype: str

In [14]:
# Use Case 5: Geo Parsing — Split to Latitude & Longitude + Range Validation

geo_split = df['Latitude,Longitude'].astype(str).str.split(',', expand=True)

# Step 2: Convert both parts to float (strips whitespace automatically via to_numeric)
df['Latitude']  = pd.to_numeric(geo_split[0].str.strip(), errors='coerce')
df['Longitude'] = pd.to_numeric(geo_split[1].str.strip(), errors='coerce')

# Step 3: Drop the original combined column — it's no longer needed
df.drop(columns=['Latitude,Longitude'], inplace=True)

# Step 4: Validate ranges for India
# Valid Latitude  → 8.0  to 37.0  (North India to South India)
# Valid Longitude → 68.0 to 97.0  (West to East India)
lat_invalid = df[
    df['Latitude'].notnull() &
    (~df['Latitude'].between(8.0, 37.0))
]
lon_invalid = df[
    df['Longitude'].notnull() &
    (~df['Longitude'].between(68.0, 97.0))
]

print(f"Rows with Latitude out of valid India range  [8–37]  : {len(lat_invalid)}")
print(f"Rows with Longitude out of valid India range [68–97] : {len(lon_invalid)}")

# Step 5: For swapped coordinates (lat looks like lon and vice versa),
# swap them back where Latitude > 68 (clearly a longitude value)
swapped = df['Latitude'] > 68
df.loc[swapped, ['Latitude', 'Longitude']] = df.loc[swapped, ['Longitude', 'Latitude']].values
print(f"\nSwapped coordinates corrected: {swapped.sum()} rows")

# Step 6: Flag any remaining out-of-range as NaN (unrecoverable)
df.loc[~df['Latitude'].between(8.0, 37.0),  'Latitude']  = np.nan
df.loc[~df['Longitude'].between(68.0, 97.0), 'Longitude'] = np.nan

remaining_bad = df['Latitude'].isnull().sum()
print(f"Rows with unrecoverable Latitude set to NaN: {remaining_bad}")

print("\nSample Latitude/Longitude after cleaning:")
print(df[['Latitude', 'Longitude']].head(10).to_string())

Rows with Latitude out of valid India range  [8–37]  : 180
Rows with Longitude out of valid India range [68–97] : 180

Swapped coordinates corrected: 180 rows
Rows with unrecoverable Latitude set to NaN: 128

Sample Latitude/Longitude after cleaning:
   Latitude  Longitude
0   13.0324    80.1744
1   12.9747    80.2142
2   12.9182    80.3353
3   13.0930    80.0978
4   13.1965    80.3709
5   12.9182    80.3353
6   12.8732    80.1064
7   12.8732    80.1064
8   12.8686    80.2389
9   13.2296    80.2745


In [15]:
df

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,Notes_Comments,Base_Temp_C,Base_Pressure_kPa,Cal_Coefficient,Cal_Version,Latitude,Longitude
0,R9001,MET-125,2025-06-05 00:00:00,5.18,3884.0,14.620,SCMH,C843,N,NaN,Inspection done. No issues.,20.8,99.500,1.002,v1.0,13.0324,80.1744
1,R9002,MET-147,2024-12-25 14:00:00,6.87,781.0,2.930,SCMH,C726,0,NaN,Email: user812@citygas.com flagged issue,21.9,101.325,1.000,v1.0,12.9747,80.2142
2,R9003,MET-132,2024-01-27 14:00:00,-3.00,793.0,3.130,SCMH,C756,0,MAINT,NaN,21.3,101.000,1.000,v1.0,12.9182,80.3353
3,R9004,MET-131,2026-01-20 15:00:00,3.24,1130.0,4.460,SCMH,C807,no,MAINT,High consumption flagged for audit.,24.5,101.325,1.000,v1.1,13.0930,80.0978
4,R9005,MET-108,2024-12-04 11:00:00,3.51,1434.0,5.020,SCMH,C735,nO,NaN,NaN,17.8,101.000,1.000,v1.0,13.1965,80.3709
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6495,R15180,MET-105,2025-10-30 16:00:00,3.65,NaN,2.020,SCMH,C770,FALSE,NaN,Calibration check passed.,19.9,101.325,1.000,v1.1,12.9124,80.1624
6496,R15181,MET-150,2026-03-05 09:00:00,4.68,NaN,1.840,SCMH,C724,0,NaN,Customer called at +91-9853538-53538,15.7,101.325,1.000,v1.0,12.9239,80.0808
6497,R15182,MET-128,2025-11-09 15:00:00,3.24,80.0,0.300,SCMH,C866,No,MAINT,Customer C752 complained about billing.,14.4,102.500,1.000,v1.1,13.0557,80.3050
6498,R15183,MET-151,2025-06-04 11:00:00,3.14,1753.0,6.150,SCMH,C791,nO,NaN,High consumption flagged for audit.,22.7,101.325,0.998,v1.0,13.2378,80.1659


In [16]:
df['Customer_ID'].unique()

<StringArray>
[   'C843',    'C726',    'C756',    'C807',    'C735',    'C757',    'C839',
    'C850',    'C740',    'c757',
 ...
 'CID-863', 'CID-894',   'C788 ', 'CID-850',   'C786 ', 'CID-757', 'CID-895',
 'CID-851', 'CID-873',   'C873 ']
Length: 196, dtype: str

In [17]:
# Use Case 6: Duplicate Read Deduplication by (Meter_ID, TS)

print("Before dedup:", df.shape)

# To ensure survivorship
df = df.sort_values('TS')

# Step 2: Drop rows where Meter_ID + TS combination is repeated
# keep='last' means we keep the final occurrence(survivorship)
df = df.drop_duplicates(subset=['Meter_ID', 'TS'], keep='last')

# Step 3: Reset index after removing rows
df = df.reset_index(drop=True)

print("After dedup :", df.shape)
print(f"Rows removed: {6500 - len(df)}")

Before dedup: (6500, 17)
After dedup : (6311, 17)
Rows removed: 189


In [18]:
df.shape

(6311, 17)

In [19]:
# Sort by meter and time so the rolling window makes chronological sense

df = df.sort_values(['Meter_ID', 'TS']).reset_index(drop=True)
df

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,Notes_Comments,Base_Temp_C,Base_Pressure_kPa,Cal_Coefficient,Cal_Version,Latitude,Longitude
0,R12777,MET-101,2024-01-01 05:00:00,2.53,1099.0,4.150,SCMH,C863,FALSE,NaN,NaN,19.8,100.000,1.000,v1.0,12.9998,80.2055
1,R11501,MET-101,2024-01-08 10:00:00,6.55,1019.0,3.850,SCMH,C863,0,NaN,Pressure drop observed at peak hours.,14.8,103.000,1.000,v1.1,12.9998,80.2055
2,R9359,MET-101,2024-01-10 20:00:00,3.31,232.0,0.900,SCMH,C863,0,NaN,NaN,20.4,102.500,1.000,v1.0,12.9998,80.2055
3,R9950,MET-101,2024-01-15 13:00:00,6.60,477.0,1.870,SCMH,C863,N,NaN,Reverse flow detected briefly.,14.9,100.000,1.000,v1.0,12.9998,80.2055
4,R15026,MET-101,2024-01-18 08:00:00,NaN,1090.0,4.260,SCMH,C863,False,NaN,Night flow anomaly recorded.,24.8,101.325,1.010,v1.0,12.9998,80.2055
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6306,R14225,MET-160,2026-03-01 00:00:00,4.35,359.0,1.410,SCMH,c796,NO,NaN,Inspection done. No issues.,23.3,99.500,1.000,v1.0,12.9801,80.2657
6307,R11792,MET-160,2026-03-06 01:00:00,5.81,1305.0,4.950,SCMH,C796,nO,NaN,NaN,22.9,99.500,0.995,v1.0,12.9801,80.2657
6308,R11594,MET-160,2026-03-07 10:00:00,4.98,204.0,0.800,SCMH,C796,no,NaN,Contact: +91-7058514-58514 for followup,14.1,101.325,1.000,v1.0,12.9801,80.2657
6309,R13635,MET-160,2026-03-11 00:05:00,4.60,404.0,1.419,SCMH,C796,False,NaN,Pressure drop observed at peak hours.,16.7,101.325,1.000,v1.1,12.9801,80.2657


In [20]:
# Use Case 7: Outlier Detection on Pressure and Flow using Rolling IQR

# Sort by meter and time so the rolling window makes chronological sense
df = df.sort_values(['Meter_ID', 'TS']).reset_index(drop=True)

def flag_outliers(series, window=50):
    """
    For each value, look at a rolling window of 50 surrounding readings.
    If the value falls outside Q1 - 1.5*IQR or Q3 + 1.5*IQR, it is an outlier.
    min_periods=10 means we need at least 10 values in the window to calculate.
    """
    q1  = series.rolling(window, min_periods=10, center=True).quantile(0.25)
    q3  = series.rolling(window, min_periods=10, center=True).quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Apply outlier flagging per meter group
df['Flow_Outlier']     = df.groupby('Meter_ID')['Flow'].transform(flag_outliers)
df['Pressure_Outlier'] = df.groupby('Meter_ID')['Pressure'].transform(flag_outliers)

print("Outlier flags added.")
print(f"  Flow outliers flagged    : {df['Flow_Outlier'].sum()}")
print(f"  Pressure outliers flagged: {df['Pressure_Outlier'].sum()}")
print("\nNote: These rows are flagged, NOT deleted.")
print("They will be excluded from forecasting but kept in the dataset for audit.")

Outlier flags added.
  Flow outliers flagged    : 100
  Pressure outliers flagged: 265

Note: These rows are flagged, NOT deleted.
They will be excluded from forecasting but kept in the dataset for audit.


In [21]:
df.value_counts()


Reading_ID  Meter_ID  TS                   Pressure  Flow    Energy  Unit  Customer_ID  Leak_Flag  Maintenance_Flag  Notes_Comments                            Base_Temp_C  Base_Pressure_kPa  Cal_Coefficient  Cal_Version  Latitude  Longitude  Flow_Outlier  Pressure_Outlier
R14085      MET-101   2024-01-20 02:00:00  6.66      301.0   1.20    SCMH  C863         False      SHUTDOWN          Maintenance done by technician ID T735.   20.6         101.325            0.998            v1.0         12.9998   80.2055    False         False               1
R12660      MET-101   2024-02-27 02:00:00  3.50      64.0    0.23    SCMH  C863         false      MAINT             Odorant alarm triggered. Cleared.         16.5         101.325            1.000            v1.0         12.9998   80.2055    False         False               1
R9609       MET-101   2024-03-12 16:00:00  3.67      138.0   0.49    SCMH  C863         No         SHUTDOWN          Pressure drop observed at peak hours.     18.3        

In [22]:
# Use Case 8: Missing Read Imputation using Last-Observation-Carried-Forward (LOCF)

print("Null counts BEFORE imputation:")
print(f"  Flow  : {df['Flow'].isnull().sum()}")
print(f"  Energy: {df['Energy'].isnull().sum()}")

# Ensure data is sorted by Meter and Time before forward-fill
df = df.sort_values(['Meter_ID', 'TS']).reset_index(drop=True)

# ffill() carries the last valid value forward within each meter group
df['Flow']   = df.groupby('Meter_ID')['Flow'].ffill()
df['Energy'] = df.groupby('Meter_ID')['Energy'].ffill()

print("\nNull counts AFTER imputation:")
print(f"  Flow  : {df['Flow'].isnull().sum()}")
print(f"  Energy: {df['Energy'].isnull().sum()}")

# Note: if a meter's very first reading is null, ffill cannot help (nothing before it)
# Those will remain as NaN — that's acceptable


Null counts BEFORE imputation:
  Flow  : 247
  Energy: 254

Null counts AFTER imputation:
  Flow  : 2
  Energy: 2


In [23]:
import re

# ── What's dirty in this dataset ──────────────────────────
# 'CID-726'  → wrong prefix, should be 'C726'
# 'c757'     → lowercase, should be 'C757'
# 'C701 '    → trailing whitespace, should be 'C701'
# All three variants refer to the same real customer

def standardize_customer_id(cid):
    if pd.isnull(cid):
        return np.nan

    cid = str(cid).strip().upper()       # strip spaces, uppercase

    # 'CID-726' → 'C726'  (remove the 'ID-' part, keep digits)
    match = re.match(r'^CID-(\d+)$', cid)
    if match:
        return 'C' + match.group(1)

    return cid                           # already clean e.g. 'C726'

# Apply to column
df['Customer_ID'] = df['Customer_ID'].apply(standardize_customer_id)

# ── Verify ─────────────────────────────────────────────────
print("Unique Customer_IDs after cleaning:", df['Customer_ID'].nunique())
print("\nAll unique values:")
print(sorted(df['Customer_ID'].unique()))

# Check if any non-standard IDs remain
not_standard = df[~df['Customer_ID'].str.match(r'^C\d{3,4}$', na=False)]
print(f"\nNon-standard IDs remaining: {len(not_standard)}")

Unique Customer_IDs after cleaning: 51

All unique values:
['C701', 'C706', 'C707', 'C708', 'C711', 'C722', 'C723', 'C724', 'C726', 'C728', 'C731', 'C735', 'C739', 'C740', 'C750', 'C755', 'C756', 'C757', 'C759', 'C762', 'C767', 'C770', 'C771', 'C786', 'C787', 'C788', 'C791', 'C796', 'C797', 'C807', 'C808', 'C814', 'C817', 'C829', 'C837', 'C839', 'C843', 'C850', 'C851', 'C854', 'C863', 'C866', 'C873', 'C878', 'C879', 'C883', 'C886', 'C888', 'C889', 'C894', 'C895']

Non-standard IDs remaining: 0


In [24]:
# ── STEP 2A: Build the Customer Master Table ───────────────

# Make sure TS is datetime (should be from UC1, double-checking)
df['TS'] = pd.to_datetime(df['TS'], errors='coerce')

# Find the earliest and latest reading date per customer
customer_dates = df.groupby('Customer_ID')['TS'].agg(
    Join_Date = 'min',
    Last_Seen = 'max'
).reset_index()

# Find the most frequently associated Meter_ID per customer
customer_meter = df.groupby('Customer_ID')['Meter_ID'].agg(
    lambda x: x.mode()[0]           # mode = most common value
).reset_index()
customer_meter.columns = ['Customer_ID', 'Meter_ID']

# Merge dates and meter info together
customer_master = customer_dates.merge(customer_meter, on='Customer_ID')

# Add Customer_Type based on customer number
# (In a real project this comes from the CRM/billing system)
# Here we assign based on ID range as a realistic simulation
def assign_customer_type(cid):
    num = int(re.search(r'\d+', cid).group())
    if num <= 740:
        return 'domestic'
    elif num <= 860:
        return 'commercial'
    else:
        return 'industrial'

customer_master['Customer_Type'] = customer_master['Customer_ID'].apply(assign_customer_type)

# Is_Active: True if last reading was within 6 months of the latest date in dataset
latest_date = df['TS'].max()
cutoff = latest_date - pd.DateOffset(months=6)
customer_master['Is_Active'] = customer_master['Last_Seen'] >= cutoff

# Keep only the 5 columns we need (drop Last_Seen, it was just for Is_Active)
customer_master = customer_master[[
    'Customer_ID', 'Meter_ID', 'Customer_Type', 'Join_Date', 'Is_Active'
]]

# Format Join_Date to date only (no time needed)
customer_master['Join_Date'] = customer_master['Join_Date'].dt.date

print(f"Customer Master Table: {customer_master.shape}")
print(f"\nPreview:")
print(customer_master.head(10).to_string(index=False))

# Save as a separate file
customer_master.to_csv('Customer_Master.csv', index=False)
print("\nSaved → Customer_Master.csv")

Customer Master Table: (51, 5)

Preview:
Customer_ID Meter_ID Customer_Type  Join_Date  Is_Active
       C701  MET-136      domestic 2024-02-12      False
       C706  MET-103      domestic 2024-01-03       True
       C707  MET-118      domestic 2024-01-03       True
       C708  MET-117      domestic 2024-01-02      False
       C711  MET-155      domestic 2024-01-02       True
       C722  MET-114      domestic 2024-01-15      False
       C723  MET-119      domestic 2024-01-09      False
       C724  MET-150      domestic 2024-01-01       True
       C726  MET-110      domestic 2024-01-02       True
       C728  MET-102      domestic 2024-01-13      False

Saved → Customer_Master.csv


In [25]:
# ── STEP 2B: Merge Master Table into Main Dataset ──────────

# Left join — every row in df is kept
# Rows whose Customer_ID exists in master → get filled columns
# Any ID NOT in master → NaN in those columns (flags a bad ID)

df = df.merge(
    customer_master[['Customer_ID', 'Customer_Type', 'Join_Date', 'Is_Active']],
    on='Customer_ID',
    how='left'
)

# Check if any rows didn't match (unrecognized Customer_ID)
unmatched = df[df['Customer_Type'].isnull()]
print(f"Rows with unrecognized Customer_ID (no master record): {len(unmatched)}")
print(f"All customers validated: {df['Customer_Type'].notnull().all()}")

print(f"\nDataset shape after merge: {df.shape}")
print("\nNew columns added:", ['Customer_Type', 'Join_Date', 'Is_Active'])
print("\nPreview:")
df[['Customer_ID', 'Customer_Type', 'Join_Date', 'Is_Active']].drop_duplicates().head(10)

Rows with unrecognized Customer_ID (no master record): 0
All customers validated: True

Dataset shape after merge: (6311, 22)

New columns added: ['Customer_Type', 'Join_Date', 'Is_Active']

Preview:


,Customer_ID,Customer_Type,Join_Date,Is_Active
0,C863,industrial,2024-01-01,False
101,C728,domestic,2024-01-13,False
207,C706,domestic,2024-01-03,True
321,C889,industrial,2024-01-02,False
423,C770,commercial,2024-01-09,False
522,C762,commercial,2024-01-06,False
625,C757,commercial,2024-01-03,False
725,C735,domestic,2024-01-08,False
835,C888,industrial,2024-01-05,False
931,C726,domestic,2024-01-02,True


In [26]:
df

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,...,Base_Pressure_kPa,Cal_Coefficient,Cal_Version,Latitude,Longitude,Flow_Outlier,Pressure_Outlier,Customer_Type,Join_Date,Is_Active
0,R12777,MET-101,2024-01-01 05:00:00,2.53,1099.0,4.150,SCMH,C863,FALSE,NaN,...,100.000,1.000,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False
1,R11501,MET-101,2024-01-08 10:00:00,6.55,1019.0,3.850,SCMH,C863,0,NaN,...,103.000,1.000,v1.1,12.9998,80.2055,False,False,industrial,2024-01-01,False
2,R9359,MET-101,2024-01-10 20:00:00,3.31,232.0,0.900,SCMH,C863,0,NaN,...,102.500,1.000,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False
3,R9950,MET-101,2024-01-15 13:00:00,6.60,477.0,1.870,SCMH,C863,N,NaN,...,100.000,1.000,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False
4,R15026,MET-101,2024-01-18 08:00:00,NaN,1090.0,4.260,SCMH,C863,False,NaN,...,101.325,1.010,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6306,R14225,MET-160,2026-03-01 00:00:00,4.35,359.0,1.410,SCMH,C796,NO,NaN,...,99.500,1.000,v1.0,12.9801,80.2657,False,False,commercial,2024-01-28,True
6307,R11792,MET-160,2026-03-06 01:00:00,5.81,1305.0,4.950,SCMH,C796,nO,NaN,...,99.500,0.995,v1.0,12.9801,80.2657,False,False,commercial,2024-01-28,True
6308,R11594,MET-160,2026-03-07 10:00:00,4.98,204.0,0.800,SCMH,C796,no,NaN,...,101.325,1.000,v1.0,12.9801,80.2657,False,False,commercial,2024-01-28,True
6309,R13635,MET-160,2026-03-11 00:05:00,4.60,404.0,1.419,SCMH,C796,False,NaN,...,101.325,1.000,v1.1,12.9801,80.2657,False,False,commercial,2024-01-28,True


In [27]:
# Use Case 10: Leak_Flag Normalization to Yes / No

# All variants found in this dataset
yes_values = {'y', 'yes', 'YES', 'YEs', '1', 'TRUE', 'True', 'true'}
no_values  = {'n', 'no',  'NO',  'nO',  '0', 'FALSE','False','false'}

def normalize_leak_flag(val):
    """Maps all dirty variants to clean Yes or No."""
    if pd.isnull(val):
        return 'Unknown'
    v = str(val).strip()
    if v in yes_values: return 'Yes'
    if v in no_values:  return 'No'
    return 'Unknown'   # catch-all for anything unexpected

df['Leak_Flag'] = df['Leak_Flag'].apply(normalize_leak_flag)

print("Leak_Flag normalized.")
print("\nValue counts after normalization:")
print(df['Leak_Flag'].value_counts())

Leak_Flag normalized.

Value counts after normalization:
Leak_Flag
No         4708
Unknown    1242
Yes         361
Name: count, dtype: int64


In [28]:
# Use Case 11: SCADA vs AMI Time Drift Correction using NTP Offsets

import numpy as np

# ─────────────────────────────────────────────────────────
# STEP 1: Simulate the NTP_Offset_Sec column
# (Skip this block entirely when working with the 15k dataset
#  — it already has this column, so just load it normally)
# ─────────────────────────────────────────────────────────

# Realistic NTP offset distribution for a CGD network:
# 70% of readings have zero drift (well-synced meters)
# 30% have some drift ranging from ±30 to ±600 seconds

np.random.seed(42)   # fixed seed so results are reproducible

n = len(df)

# Possible offset values (in seconds) and their probabilities
offset_values = [0, 30, -30, 60, -60, 120, -120, 300, -300, 600, -600]
offset_probs  = [0.70, 0.05, 0.05, 0.04, 0.04, 0.03, 0.03, 0.02, 0.02, 0.01, 0.01]

df['NTP_Offset_Sec'] = np.random.choice(offset_values, size=n, p=offset_probs)

print("NTP_Offset_Sec column added.")
print("\nDistribution of offset values:")
print(df['NTP_Offset_Sec'].value_counts().sort_index().to_string())
print(f"\nRows with non-zero offset: {(df['NTP_Offset_Sec'] != 0).sum()}")

NTP_Offset_Sec column added.

Distribution of offset values:
NTP_Offset_Sec
-600      58
-300     142
-120     165
-60      254
-30      303
 0      4454
 30      294
 60      276
 120     174
 300     126
 600      65

Rows with non-zero offset: 1857


In [29]:

# Make sure TS is datetime before we do any arithmetic on it
df['TS'] = pd.to_datetime(df['TS'], errors='coerce')

# Store original TS so we can show the before/after difference
df['TS_Original'] = df['TS'].copy()


df['TS'] = df['TS'] + pd.to_timedelta(df['NTP_Offset_Sec'], unit='s')


corrected_rows = df[df['NTP_Offset_Sec'] != 0][['Meter_ID', 'TS_Original', 'NTP_Offset_Sec', 'TS']].head(8)

print("Sample of corrected timestamps:")
print(corrected_rows.to_string(index=False))

df.drop(columns=['TS_Original'], inplace=True)

print(f"\nNTP correction complete. TS column now holds corrected timestamps.")
print(f"Dataset shape: {df.shape}")
'''

---

## What the Before/After Looks Like

When you run the verification print, you'll see something like this:

Meter_ID   TS_Original           NTP_Offset_Sec   TS (corrected)
MET-147    2024-12-25 14:00:00   30               2024-12-25 14:00:30
MET-132    2024-01-27 14:00:00   -60              2024-01-27 13:59:00
MET-130    2025-01-24 11:00:00   120              2025-01-24 11:02:00
MET-107    2024-04-10 00:00:00   -300             2024-04-09 23:55:00


Notice the fourth row — a `-300` second offset actually pushed the timestamp back across midnight into the previous day. This is exactly why UC11 must run **after UC1** (which already fixed the `24:xx` invalid hour). If you ran them in the reverse order, UC1's rollover logic would operate on already-corrected timestamps and potentially double-count.

---

## The Right Order in Your Pipeline

UC1  → Fix timestamp formats, fix 24:xx          (structural fix)
UC11 → Apply NTP offset correction               (physical correction)
UC6  → Deduplicate on (Meter_ID, TS)             (needs clean TS to work correctly)
'''

Sample of corrected timestamps:
Meter_ID         TS_Original  NTP_Offset_Sec                  TS
 MET-101 2024-01-08 10:00:00             300 2024-01-08 10:05:00
 MET-101 2024-01-10 20:00:00              30 2024-01-10 20:00:30
 MET-101 2024-02-09 03:00:00             -60 2024-02-09 02:59:00
 MET-101 2024-02-27 02:00:00              30 2024-02-27 02:00:30
 MET-101 2024-03-08 18:00:00            -300 2024-03-08 17:55:00
 MET-101 2024-03-12 16:00:00              60 2024-03-12 16:01:00
 MET-101 2024-07-12 06:00:00             -30 2024-07-12 05:59:30
 MET-101 2024-10-20 06:00:00             300 2024-10-20 06:05:00

NTP correction complete. TS column now holds corrected timestamps.
Dataset shape: (6311, 23)


"\n\n---\n\n## What the Before/After Looks Like\n\nWhen you run the verification print, you'll see something like this:\n\nMeter_ID   TS_Original           NTP_Offset_Sec   TS (corrected)\nMET-147    2024-12-25 14:00:00   30               2024-12-25 14:00:30\nMET-132    2024-01-27 14:00:00   -60              2024-01-27 13:59:00\nMET-130    2025-01-24 11:00:00   120              2025-01-24 11:02:00\nMET-107    2024-04-10 00:00:00   -300             2024-04-09 23:55:00\n\n\nNotice the fourth row — a `-300` second offset actually pushed the timestamp back across midnight into the previous day. This is exactly why UC11 must run **after UC1** (which already fixed the `24:xx` invalid hour). If you ran them in the reverse order, UC1's rollover logic would operate on already-corrected timestamps and potentially double-count.\n\n---\n\n## The Right Order in Your Pipeline\n\nUC1  → Fix timestamp formats, fix 24:xx          (structural fix)\nUC11 → Apply NTP offset correction               (physi

In [30]:
df

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,...,Cal_Coefficient,Cal_Version,Latitude,Longitude,Flow_Outlier,Pressure_Outlier,Customer_Type,Join_Date,Is_Active,NTP_Offset_Sec
0,R12777,MET-101,2024-01-01 05:00:00,2.53,1099.0,4.150,SCMH,C863,No,NaN,...,1.000,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,0
1,R11501,MET-101,2024-01-08 10:05:00,6.55,1019.0,3.850,SCMH,C863,No,NaN,...,1.000,v1.1,12.9998,80.2055,False,False,industrial,2024-01-01,False,300
2,R9359,MET-101,2024-01-10 20:00:30,3.31,232.0,0.900,SCMH,C863,No,NaN,...,1.000,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,30
3,R9950,MET-101,2024-01-15 13:00:00,6.60,477.0,1.870,SCMH,C863,Unknown,NaN,...,1.000,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,0
4,R15026,MET-101,2024-01-18 08:00:00,NaN,1090.0,4.260,SCMH,C863,No,NaN,...,1.010,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6306,R14225,MET-160,2026-03-01 00:00:00,4.35,359.0,1.410,SCMH,C796,No,NaN,...,1.000,v1.0,12.9801,80.2657,False,False,commercial,2024-01-28,True,0
6307,R11792,MET-160,2026-03-06 01:00:00,5.81,1305.0,4.950,SCMH,C796,No,NaN,...,0.995,v1.0,12.9801,80.2657,False,False,commercial,2024-01-28,True,0
6308,R11594,MET-160,2026-03-07 10:00:00,4.98,204.0,0.800,SCMH,C796,No,NaN,...,1.000,v1.0,12.9801,80.2657,False,False,commercial,2024-01-28,True,0
6309,R13635,MET-160,2026-03-11 00:06:00,4.60,404.0,1.419,SCMH,C796,No,NaN,...,1.000,v1.1,12.9801,80.2657,False,False,commercial,2024-01-28,True,60


In [31]:
df['NTP_Offset_Sec'].value_counts()

NTP_Offset_Sec
 0      4454
-30      303
 30      294
 60      276
-60      254
 120     174
-120     165
-300     142
 300     126
 600      65
-600      58
Name: count, dtype: int64

In [32]:
# USE CASE 12 : Asset mapping validation (Meter_ID ↔ Customer_ID) against installation history.

# Step 1 — Ensure dataset is sorted

df = df.sort_values(['Meter_ID', 'TS']).reset_index(drop=True)
df.head(40)

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,...,Cal_Coefficient,Cal_Version,Latitude,Longitude,Flow_Outlier,Pressure_Outlier,Customer_Type,Join_Date,Is_Active,NTP_Offset_Sec
0,R12777,MET-101,2024-01-01 05:00:00,2.53,1099.0,4.150,SCMH,C863,No,NaN,...,1.000,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,0
1,R11501,MET-101,2024-01-08 10:05:00,6.55,1019.0,3.850,SCMH,C863,No,NaN,...,1.000,v1.1,12.9998,80.2055,False,False,industrial,2024-01-01,False,300
2,R9359,MET-101,2024-01-10 20:00:30,3.31,232.0,0.900,SCMH,C863,No,NaN,...,1.000,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,30
3,R9950,MET-101,2024-01-15 13:00:00,6.60,477.0,1.870,SCMH,C863,Unknown,NaN,...,1.000,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,0
4,R15026,MET-101,2024-01-18 08:00:00,NaN,1090.0,4.260,SCMH,C863,No,NaN,...,1.010,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,0
5,R14085,MET-101,2024-01-20 02:00:00,6.66,301.0,1.200,SCMH,C863,No,SHUTDOWN,...,0.998,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,0
6,R13244,MET-101,2024-02-01 09:00:00,2.94,1204.0,4.650,SCMH,C863,Unknown,NaN,...,1.000,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,0
7,R11659,MET-101,2024-02-09 02:59:00,5.86,1204.0,3.240,SCMH,C863,Unknown,NaN,...,1.000,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,-60
8,R10878,MET-101,2024-02-15 01:00:00,7.00,995.0,3.860,SCMH,C863,Unknown,NaN,...,1.005,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,0
9,R12660,MET-101,2024-02-27 02:00:30,3.50,64.0,0.230,SCMH,C863,No,MAINT,...,1.000,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,30


In [33]:
df['Meter_ID'].unique()

<StringArray>
['MET-101', 'MET-102', 'MET-103', 'MET-104', 'MET-105', 'MET-106', 'MET-107',
 'MET-108', 'MET-109', 'MET-110', 'MET-111', 'MET-112', 'MET-113', 'MET-114',
 'MET-115', 'MET-116', 'MET-117', 'MET-118', 'MET-119', 'MET-120', 'MET-121',
 'MET-122', 'MET-123', 'MET-124', 'MET-125', 'MET-126', 'MET-127', 'MET-128',
 'MET-129', 'MET-130', 'MET-131', 'MET-132', 'MET-133', 'MET-134', 'MET-135',
 'MET-136', 'MET-137', 'MET-138', 'MET-139', 'MET-140', 'MET-141', 'MET-142',
 'MET-143', 'MET-144', 'MET-145', 'MET-146', 'MET-147', 'MET-148', 'MET-149',
 'MET-150', 'MET-151', 'MET-152', 'MET-153', 'MET-154', 'MET-155', 'MET-156',
 'MET-157', 'MET-158', 'MET-159', 'MET-160']
Length: 60, dtype: str

In [34]:
# Step 2 - Count Meter–Customer combinations

# Counts how many times each Meter_ID + Customer_ID pair occurs.

pair_counts = df.groupby(['Meter_ID', 'Customer_ID']).size()

In [35]:
pair_counts.unique()

array([101, 106, 114, 102,  99, 103, 100, 110,  96, 115,  89,  98, 116,
        92, 109, 118, 121, 105,  94, 111, 108, 122,  95, 120, 117, 107,
        87, 104, 125,  81, 131,  91,  97,  90])

In [36]:
pair_counts.head(20)

Meter_ID  Customer_ID
MET-101   C863           101
MET-102   C728           106
MET-103   C706           114
MET-104   C889           102
MET-105   C770            99
MET-106   C762           103
MET-107   C757           100
MET-108   C735           110
MET-109   C888            96
MET-110   C726           115
MET-111   C873            89
MET-112   C889            98
MET-113   C839           116
MET-114   C722            92
MET-115   C851           102
MET-116   C808           114
MET-117   C708           109
MET-118   C707           106
MET-119   C723           118
MET-120   C755           102
dtype: int64

In [37]:
# Step 3 - Create temporary Asset_Pair column

df['Asset_Pair'] = list(zip(df['Meter_ID'], df['Customer_ID']))

In [38]:
df[['Meter_ID','Customer_ID']].head()

,Meter_ID,Customer_ID
0,MET-101,C863
1,MET-101,C863
2,MET-101,C863
3,MET-101,C863
4,MET-101,C863


In [39]:
# Step 4 - Validate asset mapping

# LOGIC:-
# if pair_count > 1 → Valid mapping
# if pair_count = 1 → Suspicious mapping

df['Asset_Valid'] = df['Asset_Pair'].map(pair_counts) > 1

In [40]:
df[['Meter_ID','Customer_ID','Asset_Valid']].head(150)

,Meter_ID,Customer_ID,Asset_Valid
0,MET-101,C863,True
1,MET-101,C863,True
2,MET-101,C863,True
3,MET-101,C863,True
4,MET-101,C863,True
...,...,...,...
145,MET-102,C728,True
146,MET-102,C728,True
147,MET-102,C728,True
148,MET-102,C728,True


In [41]:
# Step 5 - Remove helper column

df.drop(columns=['Asset_Pair'], inplace=True)

In [42]:
df.head()

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,...,Cal_Version,Latitude,Longitude,Flow_Outlier,Pressure_Outlier,Customer_Type,Join_Date,Is_Active,NTP_Offset_Sec,Asset_Valid
0,R12777,MET-101,2024-01-01 05:00:00,2.53,1099.0,4.15,SCMH,C863,No,NaN,...,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,0,True
1,R11501,MET-101,2024-01-08 10:05:00,6.55,1019.0,3.85,SCMH,C863,No,NaN,...,v1.1,12.9998,80.2055,False,False,industrial,2024-01-01,False,300,True
2,R9359,MET-101,2024-01-10 20:00:30,3.31,232.0,0.90,SCMH,C863,No,NaN,...,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,30,True
3,R9950,MET-101,2024-01-15 13:00:00,6.60,477.0,1.87,SCMH,C863,Unknown,NaN,...,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,0,True
4,R15026,MET-101,2024-01-18 08:00:00,NaN,1090.0,4.26,SCMH,C863,No,NaN,...,v1.0,12.9998,80.2055,False,False,industrial,2024-01-01,False,0,True


In [43]:
# Step 6 - Inspect suspicious mappings

invalid_assets = df[df['Asset_Valid'] == False]

In [44]:
invalid_assets[['Meter_ID','Customer_ID']].drop_duplicates().head(20)

,Meter_ID,Customer_ID


In [45]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6311 entries, 0 to 6310
Data columns (total 24 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Reading_ID         6311 non-null   str           
 1   Meter_ID           6311 non-null   str           
 2   TS                 6311 non-null   datetime64[us]
 3   Pressure           6040 non-null   float64       
 4   Flow               6309 non-null   float64       
 5   Energy             6309 non-null   float64       
 6   Unit               6311 non-null   str           
 7   Customer_ID        6311 non-null   str           
 8   Leak_Flag          6311 non-null   str           
 9   Maintenance_Flag   1750 non-null   str           
 10  Notes_Comments     5898 non-null   str           
 11  Base_Temp_C        6311 non-null   float64       
 12  Base_Pressure_kPa  6311 non-null   float64       
 13  Cal_Coefficient    6311 non-null   float64       
 14  Cal_Version        

In [46]:
# Use Case 13 — Negative flow/energy checks and correction for direction flags.

In [47]:
# Step 1 — Detect negative values

df['Negative_Flow'] = df['Flow'] < 0
df['Negative_Energy'] = df['Energy'] < 0

In [48]:
# Step 2 — Create direction column

# Gas direction can be:
# Forward → normal consumption
# Reverse → backflow or sensor issue

df['Direction_Flag'] = 'Forward'
df.loc[df['Flow'] < 0, 'Direction_Flag'] = 'Reverse'

In [49]:
# Step 3 — Correct negative readings

df['Flow'] = df['Flow'].abs()
df['Energy'] = df['Energy'].abs()

In [50]:
# Step 4 — Verification

print("Negative Flow readings:", df['Negative_Flow'].sum())
print("Negative Energy readings:", df['Negative_Energy'].sum())

Negative Flow readings: 100
Negative Energy readings: 65


In [51]:
df.loc[df['Direction_Flag'] == 'Reverse', 'Flow'] = df['Flow'].abs()
df.loc[df['Direction_Flag'] == 'Reverse', 'Energy'] = df['Energy'].abs()

In [52]:
df.head(150)

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,...,Flow_Outlier,Pressure_Outlier,Customer_Type,Join_Date,Is_Active,NTP_Offset_Sec,Asset_Valid,Negative_Flow,Negative_Energy,Direction_Flag
0,R12777,MET-101,2024-01-01 05:00:00,2.53,1099.0,4.15,SCMH,C863,No,NaN,...,False,False,industrial,2024-01-01,False,0,True,False,False,Forward
1,R11501,MET-101,2024-01-08 10:05:00,6.55,1019.0,3.85,SCMH,C863,No,NaN,...,False,False,industrial,2024-01-01,False,300,True,False,False,Forward
2,R9359,MET-101,2024-01-10 20:00:30,3.31,232.0,0.90,SCMH,C863,No,NaN,...,False,False,industrial,2024-01-01,False,30,True,False,False,Forward
3,R9950,MET-101,2024-01-15 13:00:00,6.60,477.0,1.87,SCMH,C863,Unknown,NaN,...,False,False,industrial,2024-01-01,False,0,True,False,False,Forward
4,R15026,MET-101,2024-01-18 08:00:00,NaN,1090.0,4.26,SCMH,C863,No,NaN,...,False,False,industrial,2024-01-01,False,0,True,False,False,Forward
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,R11080,MET-102,2024-12-01 12:00:00,3.72,1935.0,7.02,SCMH,C728,No,NaN,...,False,False,domestic,2024-01-13,False,0,True,False,False,Forward
146,R13813,MET-102,2024-12-01 19:00:00,2.75,1327.0,5.19,SCMH,C728,No,NaN,...,False,False,domestic,2024-01-13,False,0,True,False,False,Forward
147,R13200,MET-102,2024-12-09 04:00:00,-6.68,188.0,0.67,SCMH,C728,No,NaN,...,False,True,domestic,2024-01-13,False,0,True,False,False,Forward
148,R12675,MET-102,2025-01-02 00:00:00,4.06,2387.0,0.67,SCMH,C728,No,NaN,...,False,False,domestic,2024-01-13,False,0,True,False,False,Forward


In [53]:
df['Flow'].unique()

array([1099., 1019.,  232., ...,  304.,   96.,   82.], shape=(2829,))

In [54]:
# USE CASE 14 : Calibration coefficient application and version control.

In [55]:
df.head()

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,...,Flow_Outlier,Pressure_Outlier,Customer_Type,Join_Date,Is_Active,NTP_Offset_Sec,Asset_Valid,Negative_Flow,Negative_Energy,Direction_Flag
0,R12777,MET-101,2024-01-01 05:00:00,2.53,1099.0,4.15,SCMH,C863,No,NaN,...,False,False,industrial,2024-01-01,False,0,True,False,False,Forward
1,R11501,MET-101,2024-01-08 10:05:00,6.55,1019.0,3.85,SCMH,C863,No,NaN,...,False,False,industrial,2024-01-01,False,300,True,False,False,Forward
2,R9359,MET-101,2024-01-10 20:00:30,3.31,232.0,0.90,SCMH,C863,No,NaN,...,False,False,industrial,2024-01-01,False,30,True,False,False,Forward
3,R9950,MET-101,2024-01-15 13:00:00,6.60,477.0,1.87,SCMH,C863,Unknown,NaN,...,False,False,industrial,2024-01-01,False,0,True,False,False,Forward
4,R15026,MET-101,2024-01-18 08:00:00,NaN,1090.0,4.26,SCMH,C863,No,NaN,...,False,False,industrial,2024-01-01,False,0,True,False,False,Forward


In [56]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6311 entries, 0 to 6310
Data columns (total 27 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Reading_ID         6311 non-null   str           
 1   Meter_ID           6311 non-null   str           
 2   TS                 6311 non-null   datetime64[us]
 3   Pressure           6040 non-null   float64       
 4   Flow               6309 non-null   float64       
 5   Energy             6309 non-null   float64       
 6   Unit               6311 non-null   str           
 7   Customer_ID        6311 non-null   str           
 8   Leak_Flag          6311 non-null   str           
 9   Maintenance_Flag   1750 non-null   str           
 10  Notes_Comments     5898 non-null   str           
 11  Base_Temp_C        6311 non-null   float64       
 12  Base_Pressure_kPa  6311 non-null   float64       
 13  Cal_Coefficient    6311 non-null   float64       
 14  Cal_Version        

In [57]:
df.columns

Index(['Reading_ID', 'Meter_ID', 'TS', 'Pressure', 'Flow', 'Energy', 'Unit',
       'Customer_ID', 'Leak_Flag', 'Maintenance_Flag', 'Notes_Comments',
       'Base_Temp_C', 'Base_Pressure_kPa', 'Cal_Coefficient', 'Cal_Version',
       'Latitude', 'Longitude', 'Flow_Outlier', 'Pressure_Outlier',
       'Customer_Type', 'Join_Date', 'Is_Active', 'NTP_Offset_Sec',
       'Asset_Valid', 'Negative_Flow', 'Negative_Energy', 'Direction_Flag'],
      dtype='str')

In [58]:
df['Cal_Coefficient'].nunique()

7

In [59]:
df['Cal_Coefficient'].unique()

array([1.   , 1.01 , 0.998, 1.005, 1.002, 0.995, 0.99 ])

In [60]:
df.head(20)

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,...,Flow_Outlier,Pressure_Outlier,Customer_Type,Join_Date,Is_Active,NTP_Offset_Sec,Asset_Valid,Negative_Flow,Negative_Energy,Direction_Flag
0,R12777,MET-101,2024-01-01 05:00:00,2.53,1099.0,4.15,SCMH,C863,No,NaN,...,False,False,industrial,2024-01-01,False,0,True,False,False,Forward
1,R11501,MET-101,2024-01-08 10:05:00,6.55,1019.0,3.85,SCMH,C863,No,NaN,...,False,False,industrial,2024-01-01,False,300,True,False,False,Forward
2,R9359,MET-101,2024-01-10 20:00:30,3.31,232.0,0.90,SCMH,C863,No,NaN,...,False,False,industrial,2024-01-01,False,30,True,False,False,Forward
3,R9950,MET-101,2024-01-15 13:00:00,6.60,477.0,1.87,SCMH,C863,Unknown,NaN,...,False,False,industrial,2024-01-01,False,0,True,False,False,Forward
4,R15026,MET-101,2024-01-18 08:00:00,NaN,1090.0,4.26,SCMH,C863,No,NaN,...,False,False,industrial,2024-01-01,False,0,True,False,False,Forward
5,R14085,MET-101,2024-01-20 02:00:00,6.66,301.0,1.20,SCMH,C863,No,SHUTDOWN,...,False,False,industrial,2024-01-01,False,0,True,False,False,Forward
6,R13244,MET-101,2024-02-01 09:00:00,2.94,1204.0,4.65,SCMH,C863,Unknown,NaN,...,False,False,industrial,2024-01-01,False,0,True,False,False,Forward
7,R11659,MET-101,2024-02-09 02:59:00,5.86,1204.0,3.24,SCMH,C863,Unknown,NaN,...,False,False,industrial,2024-01-01,False,-60,True,False,False,Forward
8,R10878,MET-101,2024-02-15 01:00:00,7.00,995.0,3.86,SCMH,C863,Unknown,NaN,...,False,False,industrial,2024-01-01,False,0,True,False,False,Forward
9,R12660,MET-101,2024-02-27 02:00:30,3.50,64.0,0.23,SCMH,C863,No,MAINT,...,False,False,industrial,2024-01-01,False,30,True,False,False,Forward


In [61]:
df['Cal_Coefficient'].value_counts()

Cal_Coefficient
1.000    3144
1.010     577
1.005     535
0.998     523
0.990     519
1.002     508
0.995     505
Name: count, dtype: int64

In [62]:
# Step 1 - Ensure coefficient is numeric

df['Cal_Coefficient'] = pd.to_numeric(df['Cal_Coefficient'], errors='coerce').fillna(1.000)

In [63]:
# Step 2 - Apply calibration

df['Flow_calibrated'] = (df['Flow'] * df['Cal_Coefficient']).round(3)

df['Energy_calibrated'] = (df['Energy'] * df['Cal_Coefficient']).round(3)

In [64]:
df['Cal_Version'].value_counts()

Cal_Version
v1.0    3493
v1.1    2114
v1.2     704
Name: count, dtype: int64

In [65]:
# Step 3 - Maintain calibration version

df['Cal_Version'] = df['Cal_Version'].fillna('v1.0')

In [66]:
# Step 4 - Verify results

df[['Flow','Energy','Cal_Coefficient','Flow_calibrated','Energy_calibrated','Cal_Version']].head(10)

,Flow,Energy,Cal_Coefficient,Flow_calibrated,Energy_calibrated,Cal_Version
0,1099.0,4.15,1.000,1099.000,4.150,v1.0
1,1019.0,3.85,1.000,1019.000,3.850,v1.1
2,232.0,0.90,1.000,232.000,0.900,v1.0
3,477.0,1.87,1.000,477.000,1.870,v1.0
4,1090.0,4.26,1.010,1100.900,4.303,v1.0
5,301.0,1.20,0.998,300.398,1.198,v1.0
6,1204.0,4.65,1.000,1204.000,4.650,v1.0
7,1204.0,3.24,1.000,1204.000,3.240,v1.0
8,995.0,3.86,1.005,999.975,3.879,v1.0
9,64.0,0.23,1.000,64.000,0.230,v1.0


In [67]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6311 entries, 0 to 6310
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Reading_ID         6311 non-null   str           
 1   Meter_ID           6311 non-null   str           
 2   TS                 6311 non-null   datetime64[us]
 3   Pressure           6040 non-null   float64       
 4   Flow               6309 non-null   float64       
 5   Energy             6309 non-null   float64       
 6   Unit               6311 non-null   str           
 7   Customer_ID        6311 non-null   str           
 8   Leak_Flag          6311 non-null   str           
 9   Maintenance_Flag   1750 non-null   str           
 10  Notes_Comments     5898 non-null   str           
 11  Base_Temp_C        6311 non-null   float64       
 12  Base_Pressure_kPa  6311 non-null   float64       
 13  Cal_Coefficient    6311 non-null   float64       
 14  Cal_Version        

In [68]:
# USE CASE 15 : Event log correlation to mark maintenance windows as non‑billable.

In [69]:
df['Maintenance_Flag'].unique()

<StringArray>
[nan, 'SHUTDOWN', 'MAINT']
Length: 3, dtype: str

In [70]:
df['Maintenance_Flag'].value_counts()

Maintenance_Flag
MAINT       1157
SHUTDOWN     593
Name: count, dtype: int64

In [71]:
df['Maintenance_Flag'].isna()

0        True
1        True
2        True
3        True
4        True
        ...  
6306     True
6307     True
6308     True
6309     True
6310    False
Name: Maintenance_Flag, Length: 6311, dtype: bool

In [72]:
# Step 2 - FILL THE NULL VALUES WITH NORMAL & change from maint to maintenance

# df['Maintenance_Flag'] = df['Maintenance_Flag'].replace('NO MAINTENANCE', 'NORMAL')
df['Maintenance_Flag'] = df['Maintenance_Flag'].fillna('NORMAL')
df['Maintenance_Flag'] = df['Maintenance_Flag'].replace('MAINT', 'MAINTENANCE')

In [73]:
df['Maintenance_Flag'].value_counts()

Maintenance_Flag
NORMAL         4561
MAINTENANCE    1157
SHUTDOWN        593
Name: count, dtype: int64

In [74]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6311 entries, 0 to 6310
Data columns (total 29 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Reading_ID         6311 non-null   str           
 1   Meter_ID           6311 non-null   str           
 2   TS                 6311 non-null   datetime64[us]
 3   Pressure           6040 non-null   float64       
 4   Flow               6309 non-null   float64       
 5   Energy             6309 non-null   float64       
 6   Unit               6311 non-null   str           
 7   Customer_ID        6311 non-null   str           
 8   Leak_Flag          6311 non-null   str           
 9   Maintenance_Flag   6311 non-null   str           
 10  Notes_Comments     5898 non-null   str           
 11  Base_Temp_C        6311 non-null   float64       
 12  Base_Pressure_kPa  6311 non-null   float64       
 13  Cal_Coefficient    6311 non-null   float64       
 14  Cal_Version        

In [75]:
# Step 4 - Create billing column

df['Is_Billable'] = True

In [76]:
# Step 5 - Mark maintenance & shutdown as non-billable

non_billable_events = ['MAINTENANCE', 'SHUTDOWN']

df.loc[df['Maintenance_Flag'].isin(non_billable_events), 'Is_Billable'] = False

In [77]:
# Step 6 - Verify results

df[['Maintenance_Flag','Is_Billable']].value_counts()

Maintenance_Flag  Is_Billable
NORMAL            True           4561
MAINTENANCE       False          1157
SHUTDOWN          False           593
Name: count, dtype: int64

In [78]:
# USE CASE 16 : Pressure/temperature base condition adjustment to standard conditions.

In [79]:
df.shape

(6311, 30)

In [80]:
# Step 3 - VERIFY IF PRESENT IN THE MAIN DATASET

df.info()
df.head(150)

<class 'pandas.DataFrame'>
RangeIndex: 6311 entries, 0 to 6310
Data columns (total 30 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   Reading_ID         6311 non-null   str           
 1   Meter_ID           6311 non-null   str           
 2   TS                 6311 non-null   datetime64[us]
 3   Pressure           6040 non-null   float64       
 4   Flow               6309 non-null   float64       
 5   Energy             6309 non-null   float64       
 6   Unit               6311 non-null   str           
 7   Customer_ID        6311 non-null   str           
 8   Leak_Flag          6311 non-null   str           
 9   Maintenance_Flag   6311 non-null   str           
 10  Notes_Comments     5898 non-null   str           
 11  Base_Temp_C        6311 non-null   float64       
 12  Base_Pressure_kPa  6311 non-null   float64       
 13  Cal_Coefficient    6311 non-null   float64       
 14  Cal_Version        

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,...,Join_Date,Is_Active,NTP_Offset_Sec,Asset_Valid,Negative_Flow,Negative_Energy,Direction_Flag,Flow_calibrated,Energy_calibrated,Is_Billable
0,R12777,MET-101,2024-01-01 05:00:00,2.53,1099.0,4.15,SCMH,C863,No,NORMAL,...,2024-01-01,False,0,True,False,False,Forward,1099.000,4.150,True
1,R11501,MET-101,2024-01-08 10:05:00,6.55,1019.0,3.85,SCMH,C863,No,NORMAL,...,2024-01-01,False,300,True,False,False,Forward,1019.000,3.850,True
2,R9359,MET-101,2024-01-10 20:00:30,3.31,232.0,0.90,SCMH,C863,No,NORMAL,...,2024-01-01,False,30,True,False,False,Forward,232.000,0.900,True
3,R9950,MET-101,2024-01-15 13:00:00,6.60,477.0,1.87,SCMH,C863,Unknown,NORMAL,...,2024-01-01,False,0,True,False,False,Forward,477.000,1.870,True
4,R15026,MET-101,2024-01-18 08:00:00,NaN,1090.0,4.26,SCMH,C863,No,NORMAL,...,2024-01-01,False,0,True,False,False,Forward,1100.900,4.303,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,R11080,MET-102,2024-12-01 12:00:00,3.72,1935.0,7.02,SCMH,C728,No,NORMAL,...,2024-01-13,False,0,True,False,False,Forward,1935.000,7.020,True
146,R13813,MET-102,2024-12-01 19:00:00,2.75,1327.0,5.19,SCMH,C728,No,NORMAL,...,2024-01-13,False,0,True,False,False,Forward,1329.654,5.200,True
147,R13200,MET-102,2024-12-09 04:00:00,-6.68,188.0,0.67,SCMH,C728,No,NORMAL,...,2024-01-13,False,0,True,False,False,Forward,187.060,0.667,True
148,R12675,MET-102,2025-01-02 00:00:00,4.06,2387.0,0.67,SCMH,C728,No,NORMAL,...,2024-01-13,False,0,True,False,False,Forward,2363.130,0.663,True


In [81]:
# Step 4 - DEFINE STANDARD CONSTANTS

STD_TEMP = 15.0 # IN DEGREE CELSIUS
STD_PRESSURE = 101.325 # IN KPa

In [82]:
# Step 5 - Apply Base Condition Adjustment

# Compute standardized flow :

df['Flow_std'] = (
    df['Flow_calibrated']
    * (df['Base_Pressure_kPa'] / STD_PRESSURE)
    * ((273.15 + STD_TEMP) / (273.15 + df['Base_Temp_C']))
).round(3)

In [83]:
# Step 6 - Verify Result

df[['Flow_calibrated','Base_Temp_C','Base_Pressure_kPa','Flow_std']].head()

,Flow_calibrated,Base_Temp_C,Base_Pressure_kPa,Flow_std
0,1099.0,19.8,100.000,1066.857
1,1019.0,14.8,103.000,1036.565
2,232.0,20.4,102.500,230.373
3,477.0,14.9,100.000,470.926
4,1100.9,24.8,101.325,1064.690


In [84]:
# USE CASE 17 : Sensor health checks (flat‑line, stuck values).

In [85]:
# Step 1 — Check for Sorted dataset

df.head(150)

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,...,Is_Active,NTP_Offset_Sec,Asset_Valid,Negative_Flow,Negative_Energy,Direction_Flag,Flow_calibrated,Energy_calibrated,Is_Billable,Flow_std
0,R12777,MET-101,2024-01-01 05:00:00,2.53,1099.0,4.15,SCMH,C863,No,NORMAL,...,False,0,True,False,False,Forward,1099.000,4.150,True,1066.857
1,R11501,MET-101,2024-01-08 10:05:00,6.55,1019.0,3.85,SCMH,C863,No,NORMAL,...,False,300,True,False,False,Forward,1019.000,3.850,True,1036.565
2,R9359,MET-101,2024-01-10 20:00:30,3.31,232.0,0.90,SCMH,C863,No,NORMAL,...,False,30,True,False,False,Forward,232.000,0.900,True,230.373
3,R9950,MET-101,2024-01-15 13:00:00,6.60,477.0,1.87,SCMH,C863,Unknown,NORMAL,...,False,0,True,False,False,Forward,477.000,1.870,True,470.926
4,R15026,MET-101,2024-01-18 08:00:00,NaN,1090.0,4.26,SCMH,C863,No,NORMAL,...,False,0,True,False,False,Forward,1100.900,4.303,True,1064.690
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,R11080,MET-102,2024-12-01 12:00:00,3.72,1935.0,7.02,SCMH,C728,No,NORMAL,...,False,0,True,False,False,Forward,1935.000,7.020,True,1923.651
146,R13813,MET-102,2024-12-01 19:00:00,2.75,1327.0,5.19,SCMH,C728,No,NORMAL,...,False,0,True,False,False,Forward,1329.654,5.200,True,1322.768
147,R13200,MET-102,2024-12-09 04:00:00,-6.68,188.0,0.67,SCMH,C728,No,NORMAL,...,False,0,True,False,False,Forward,187.060,0.667,True,189.295
148,R12675,MET-102,2025-01-02 00:00:00,4.06,2387.0,0.67,SCMH,C728,No,NORMAL,...,False,0,True,False,False,Forward,2363.130,0.663,True,2269.375


In [86]:
# Step 2 — Detect repeated values

# check if values do not change compared to the previous reading.

df['Flow_same'] = df.groupby('Meter_ID')['Flow'].diff() == 0
df['Pressure_same'] = df.groupby('Meter_ID')['Pressure'].diff() == 0
df['Energy_same'] = df.groupby('Meter_ID')['Energy'].diff() == 0

In [87]:
# Step 3 — Detect flat-line sensors

# flag sensors where all three remain unchanged.

df['Sensor_Stuck'] = (
    df['Flow_same'] &
    df['Pressure_same'] &
    df['Energy_same']
)

In [88]:
# Step 4 — Final Sensor Health Column

df['Sensor_Reliable'] = ~df['Sensor_Stuck']

In [89]:
df['Sensor_Reliable'].value_counts()

Sensor_Reliable
True    6311
Name: count, dtype: int64

In [90]:
# Step 5 — Verify results

df[['Meter_ID','TS','Flow','Pressure','Energy','Sensor_Stuck']].head(10)

,Meter_ID,TS,Flow,Pressure,Energy,Sensor_Stuck
0,MET-101,2024-01-01 05:00:00,1099.0,2.53,4.15,False
1,MET-101,2024-01-08 10:05:00,1019.0,6.55,3.85,False
2,MET-101,2024-01-10 20:00:30,232.0,3.31,0.90,False
3,MET-101,2024-01-15 13:00:00,477.0,6.60,1.87,False
4,MET-101,2024-01-18 08:00:00,1090.0,NaN,4.26,False
5,MET-101,2024-01-20 02:00:00,301.0,6.66,1.20,False
6,MET-101,2024-02-01 09:00:00,1204.0,2.94,4.65,False
7,MET-101,2024-02-09 02:59:00,1204.0,5.86,3.24,False
8,MET-101,2024-02-15 01:00:00,995.0,7.00,3.86,False
9,MET-101,2024-02-27 02:00:30,64.0,3.50,0.23,False


In [91]:
df['Sensor_Stuck'].sum()

np.int64(0)

In [92]:
# USE CASE 18 : Reading plausibility by meter capacity.

In [93]:
# Step 1 — Define capacity limits

capacity_map = {
    'domestic': 10,
    'commercial': 100,
    'industrial': 500
}

In [94]:
# Step 2 — Create capacity column

df['Meter_Capacity_SCMH'] = df['Customer_Type'].map(capacity_map)

In [95]:
# Step 3 — Check plausibility

df['Capacity_Exceeded'] = df['Flow_calibrated'] > df['Meter_Capacity_SCMH']

In [96]:
# Step 4 — Create plausibility flag

df['Reading_Plausible'] = ~df['Capacity_Exceeded']

In [97]:
# Step 5 — Verify

df[['Customer_Type','Flow_calibrated','Meter_Capacity_SCMH','Reading_Plausible']].head(10)

,Customer_Type,Flow_calibrated,Meter_Capacity_SCMH,Reading_Plausible
0,industrial,1099.000,500,False
1,industrial,1019.000,500,False
2,industrial,232.000,500,True
3,industrial,477.000,500,True
4,industrial,1100.900,500,False
5,industrial,300.398,500,True
6,industrial,1204.000,500,False
7,industrial,1204.000,500,False
8,industrial,999.975,500,False
9,industrial,64.000,500,True


In [98]:
df['Capacity_Exceeded'].sum()

np.int64(5948)

In [99]:
# USE CASE 19 : Backfill missing units based on meter profile.

In [100]:
print("Missing Unit values:", df['Unit'].isnull().sum())

Missing Unit values: 0


In [101]:
unit_map = {
    'domestic': 'SCMH',
    'commercial': 'SCMH',
    'industrial': 'SCMH'
}

In [102]:
df['Unit'] = df.apply(
    lambda row: unit_map.get(row['Customer_Type'], 'SCMH')
    if pd.isna(row['Unit']) else row['Unit'],
    axis=1
)

In [103]:
print("Remaining missing units:", df['Unit'].isnull().sum())
print(df['Unit'].value_counts())

Remaining missing units: 0
Unit
SCMH    6311
Name: count, dtype: int64


In [104]:
print("Missing Unit values:", df['Unit'].isnull().sum())

Missing Unit values: 0


In [105]:
df.sample(150)

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,...,Is_Billable,Flow_std,Flow_same,Pressure_same,Energy_same,Sensor_Stuck,Sensor_Reliable,Meter_Capacity_SCMH,Capacity_Exceeded,Reading_Plausible
3538,R9708,MET-134,2024-05-25 07:00:00,4.59,726.0,2.66,SCMH,C850,No,NORMAL,...,True,729.477,False,False,False,False,True,100,True,False
2054,R12714,MET-120,2025-04-16 08:59:00,2.61,2300.0,8.18,SCMH,C755,No,NORMAL,...,True,2190.904,False,False,False,False,True,100,True,False
1869,R12192,MET-118,2026-03-08 15:00:30,55.50,1256.0,5.01,SCMH,C707,No,NORMAL,...,True,1244.225,False,False,False,False,True,10,True,False
4492,R14870,MET-143,2024-03-22 04:00:00,4.35,2200.0,6.76,SCMH,C739,No,NORMAL,...,True,2182.184,False,False,False,False,True,10,True,False
4971,R14775,MET-147,2025-07-19 10:02:00,6.41,1953.0,7.67,SCMH,C726,No,NORMAL,...,True,1911.863,False,False,False,False,True,10,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4162,R14281,MET-139,2025-11-08 03:00:00,4.44,411.0,1.62,SCMH,C878,Unknown,NORMAL,...,True,403.028,False,False,False,False,True,500,False,True
5381,R10049,MET-151,2025-02-28 06:00:00,3.92,1821.0,6.70,SCMH,C791,No,NORMAL,...,True,1810.134,False,False,False,False,True,100,True,False
3982,R14048,MET-138,2024-06-26 06:00:00,6.42,53000.0,0.20,SCMH,C740,No,SHUTDOWN,...,False,51767.964,False,False,False,False,True,10,True,False
1061,R10162,MET-111,2024-06-29 18:10:00,3.82,814.0,2.86,SCMH,C873,No,SHUTDOWN,...,False,818.135,False,False,False,False,True,500,True,False


In [106]:
# USE CASE 20 : PII masking in notes and comments.

In [107]:
# Step 3 - VERIFY IF PRESENT IN THE MAIN DATASET

df.info()
df.head(150)

<class 'pandas.DataFrame'>
RangeIndex: 6311 entries, 0 to 6310
Data columns (total 39 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Reading_ID           6311 non-null   str           
 1   Meter_ID             6311 non-null   str           
 2   TS                   6311 non-null   datetime64[us]
 3   Pressure             6040 non-null   float64       
 4   Flow                 6309 non-null   float64       
 5   Energy               6309 non-null   float64       
 6   Unit                 6311 non-null   str           
 7   Customer_ID          6311 non-null   str           
 8   Leak_Flag            6311 non-null   str           
 9   Maintenance_Flag     6311 non-null   str           
 10  Notes_Comments       5898 non-null   str           
 11  Base_Temp_C          6311 non-null   float64       
 12  Base_Pressure_kPa    6311 non-null   float64       
 13  Cal_Coefficient      6311 non-null   float64

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,...,Is_Billable,Flow_std,Flow_same,Pressure_same,Energy_same,Sensor_Stuck,Sensor_Reliable,Meter_Capacity_SCMH,Capacity_Exceeded,Reading_Plausible
0,R12777,MET-101,2024-01-01 05:00:00,2.53,1099.0,4.15,SCMH,C863,No,NORMAL,...,True,1066.857,False,False,False,False,True,500,True,False
1,R11501,MET-101,2024-01-08 10:05:00,6.55,1019.0,3.85,SCMH,C863,No,NORMAL,...,True,1036.565,False,False,False,False,True,500,True,False
2,R9359,MET-101,2024-01-10 20:00:30,3.31,232.0,0.90,SCMH,C863,No,NORMAL,...,True,230.373,False,False,False,False,True,500,False,True
3,R9950,MET-101,2024-01-15 13:00:00,6.60,477.0,1.87,SCMH,C863,Unknown,NORMAL,...,True,470.926,False,False,False,False,True,500,False,True
4,R15026,MET-101,2024-01-18 08:00:00,NaN,1090.0,4.26,SCMH,C863,No,NORMAL,...,True,1064.690,False,False,False,False,True,500,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
145,R11080,MET-102,2024-12-01 12:00:00,3.72,1935.0,7.02,SCMH,C728,No,NORMAL,...,True,1923.651,False,False,False,False,True,10,True,False
146,R13813,MET-102,2024-12-01 19:00:00,2.75,1327.0,5.19,SCMH,C728,No,NORMAL,...,True,1322.768,False,False,False,False,True,10,True,False
147,R13200,MET-102,2024-12-09 04:00:00,-6.68,188.0,0.67,SCMH,C728,No,NORMAL,...,True,189.295,False,False,False,False,True,10,True,False
148,R12675,MET-102,2025-01-02 00:00:00,4.06,2387.0,0.67,SCMH,C728,No,NORMAL,...,True,2269.375,False,False,True,False,True,10,True,False


In [108]:
# Step 4 - PII Masking on notes and comments

import re

def mask_pii(text):
    if pd.isna(text):
        return text

    text = re.sub(r'[\w\.-]+@[\w\.-]+\.\w+', '[EMAIL REDACTED]', str(text))
    text = re.sub(r'\b\d{10}\b', '[PHONE REDACTED]', text)
    text = re.sub(r'\+?\d[\d\-\s]{7,}\d', '[MISC. PHONE REDACTED]', text)
    text = re.sub(r'\bT\d{3,4}\b', '[TECH-ID REDACTED]', text)
    text = re.sub(r'\b[A-Z][a-z]+ [A-Z][a-z]+\b', '[NAME REDACTED]', text)

    return text

df['Notes_Comments'] = df['Notes_Comments'].apply(mask_pii)

In [109]:
# Step 5 - Apply to the column

df['Notes_Comments'] = df['Notes_Comments'].apply(mask_pii)

In [110]:
# Step 6 - Verify Results

df[['Notes_Comments']].head(150)
df.sample(1500)

,Reading_ID,Meter_ID,TS,Pressure,Flow,Energy,Unit,Customer_ID,Leak_Flag,Maintenance_Flag,...,Is_Billable,Flow_std,Flow_same,Pressure_same,Energy_same,Sensor_Stuck,Sensor_Reliable,Meter_Capacity_SCMH,Capacity_Exceeded,Reading_Plausible
3916,R11258,MET-137,2025-07-19 12:00:30,4.64,579.0,2.23,SCMH,C894,No,NORMAL,...,True,580.078,False,False,False,False,True,500,True,False
5471,R14800,MET-152,2024-09-19 03:00:00,3.97,721.0,2.59,SCMH,C788,No,SHUTDOWN,...,False,695.066,False,False,False,False,True,100,True,False
4261,R11513,MET-140,2026-01-14 16:02:00,6.67,2735.0,10.85,SCMH,C808,No,MAINTENANCE,...,False,2747.645,False,False,False,False,True,100,True,False
1780,R12685,MET-118,2024-05-20 18:00:00,3.41,1367.0,5.24,SCMH,C707,No,NORMAL,...,True,1310.204,False,False,False,False,True,10,True,False
100,R13190,MET-101,2026-03-11 00:05:00,6.11,382.0,1.35,SCMH,C863,No,NORMAL,...,True,370.056,False,False,False,False,True,500,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3312,R9352,MET-132,2024-07-22 05:00:30,NaN,51.0,0.20,SCMH,C756,No,NORMAL,...,True,50.770,False,False,False,False,True,100,False,True
5414,R12905,MET-151,2025-09-30 09:00:00,3.55,1299.0,4.93,SCMH,C791,No,MAINTENANCE,...,False,1266.269,False,False,False,False,True,100,True,False
2168,R9377,MET-121,2025-04-14 11:00:00,3.17,968.0,3.65,SCMH,C759,No,NORMAL,...,True,929.295,False,False,False,False,True,100,True,False
4742,R14018,MET-145,2024-10-25 15:00:00,5.52,1479.0,5.57,SCMH,C895,No,MAINTENANCE,...,False,1430.813,False,False,False,False,True,500,True,False


In [111]:
# df_raw.to_csv('CGD_MasterRaw.csv', index=False)

In [114]:
df.to_csv('CGD_MasterCleaned.csv', index=False)

In [113]:
df.shape

(6311, 39)